# Detección de Anomalías

**Universidad de los Andes — Minería de Datos**  
**Semana 10 — Detección de Anomalías**

---

## Objetivos de aprendizaje

Al finalizar este notebook, el estudiante será capaz de:

1. Definir qué es una anomalía y distinguir sus tres tipos principales (puntual, contextual, colectiva).
2. Aplicar métodos unidimensionales clásicos: Z-score, IQR y la prueba de Grubbs.
3. Implementar detectores multidimensionales: Isolation Forest, Local Outlier Factor (LOF) y One-Class SVM.
4. Evaluar detectores con métricas apropiadas para datos desbalanceados: Precision, Recall, F1 y ROC-AUC.
5. Construir un pipeline completo de detección de anomalías sobre un conjunto de datos sintético similar a fraude financiero.

---

**Sesiones cubiertas:**
- Sesión 19 — Conceptos básicos y algoritmos unidimensionales
- Sesión 20 — Algoritmos multidimensionales

---
## 1. Introducción: ¿Qué es una anomalía?

Una **anomalía** (también llamada *outlier*, valor atípico o punto anómalo) es una observación que se desvía significativamente del comportamiento esperado de los datos. La definición clásica de Hawkins (1980) es:

> *"Una observación que se aparta tanto del resto de las observaciones que se sospecha que fue generada por un mecanismo diferente."*

### ¿Por qué detectar anomalías?

La detección de anomalías tiene un valor enorme en múltiples dominios:

| Dominio | Anomalía | Consecuencia si no se detecta |
|---|---|---|
| Banca / Finanzas | Transacción fraudulenta | Pérdida económica |
| Manufactura | Fallo en sensor de máquina | Accidente o parada de producción |
| Salud | Resultado de laboratorio inusual | Diagnóstico tardío |
| Ciberseguridad | Tráfico de red no autorizado | Brecha de datos |
| Telecomunicaciones | Uso anormal de datos | Fraude en roaming |

### Reto principal: datos desbalanceados

En casi todos los casos reales, las anomalías son **raras**: menos del 1–5% de los datos. Esto hace que:
- Los modelos supervisados estándar fracasen (aprenden a predecir siempre la clase mayoritaria).
- La exactitud (*accuracy*) sea una métrica inútil (un modelo que dice «todo es normal» tiene 99% de accuracy en fraude).
- Se requieran enfoques no supervisados o semi-supervisados.

---
## 2. Tipos de anomalías

### 2.1 Anomalías puntuales (*Point anomalies*)
Una sola observación es anómala respecto al resto del conjunto de datos.  
**Ejemplo:** una transacción de \$50,000 cuando el usuario típicamente gasta \$200.

### 2.2 Anomalías contextuales (*Contextual anomalies*)
Una observación es anómala **en un contexto específico**, pero no necesariamente en otros contextos.  
**Ejemplo:** una temperatura de 30°C es normal en verano, pero anómala en diciembre en Bogotá.

### 2.3 Anomalías colectivas (*Collective anomalies*)
Un **grupo** de observaciones es anómalo respecto al conjunto completo, aunque individualmente cada punto pueda parecer normal.  
**Ejemplo:** una secuencia de 50 transacciones pequeñas consecutivas en 5 minutos (técnica de *smurfing* en lavado de dinero).

A continuación visualizamos los tres tipos con datos sintéticos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# Configuración global de gráficos
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

np.random.seed(42)
print('Librerías cargadas correctamente.')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# --- Anomalía puntual ---
datos_normales = np.random.normal(loc=200, scale=30, size=100)
anomalia_puntual = np.array([800])  # Transacción gigante
todos = np.concatenate([datos_normales, anomalia_puntual])

ax = axes[0]
ax.scatter(range(len(datos_normales)), datos_normales,
           color='steelblue', alpha=0.6, label='Normal', s=30)
ax.scatter([len(datos_normales)], anomalia_puntual,
           color='red', s=120, zorder=5, label='Anomalía', marker='*')
ax.set_title('Anomalía Puntual')
ax.set_xlabel('Índice de transacción')
ax.set_ylabel('Monto ($)')
ax.legend()

# --- Anomalía contextual ---
meses = np.arange(1, 13)
# Temperatura promedio mensual en Bogotá (°C) — patrón típico
temp_tipica = np.array([14, 14, 14, 14, 14, 13, 13, 13, 14, 14, 14, 14])
temp_real = temp_tipica.copy()
temp_real[6] = 28  # Julio anómalamente caliente

ax = axes[1]
ax.plot(meses, temp_tipica, 'o--', color='steelblue',
        alpha=0.5, label='Esperada')
ax.plot(meses, temp_real, 'o-', color='gray', label='Observada')
ax.scatter([7], [28], color='red', s=150, zorder=5,
           marker='*', label='Anomalía contextual')
ax.set_title('Anomalía Contextual')
ax.set_xlabel('Mes')
ax.set_ylabel('Temperatura (°C)')
ax.set_xticks(meses)
ax.set_xticklabels(['Ene','Feb','Mar','Abr','May','Jun',
                    'Jul','Ago','Sep','Oct','Nov','Dic'], rotation=45)
ax.legend(fontsize=8)

# --- Anomalía colectiva ---
tiempo = np.arange(0, 200)
# Comportamiento normal: montos pequeños aleatorios
montos = np.random.uniform(10, 80, size=200)
# Segmento anómalo: 20 transacciones pequeñas muy seguidas (smurfing)
montos[80:100] = np.random.uniform(9, 11, size=20)  # Importes casi idénticos

ax = axes[2]
colores = ['red' if 80 <= i < 100 else 'steelblue' for i in tiempo]
ax.bar(tiempo, montos, color=colores, alpha=0.7, width=1)
ax.set_title('Anomalía Colectiva')
ax.set_xlabel('Índice de transacción')
ax.set_ylabel('Monto ($)')
patch_normal = mpatches.Patch(color='steelblue', alpha=0.7, label='Normal')
patch_anom = mpatches.Patch(color='red', alpha=0.7, label='Colectiva (smurfing)')
ax.legend(handles=[patch_normal, patch_anom], fontsize=8)

plt.suptitle('Tipos de Anomalías', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 3. Algoritmos Unidimensionales (Sesión 19)

Los métodos unidimensionales analizan **una variable a la vez**. Son simples, interpretables y útiles como primera exploración.

Generaremos un vector de montos de transacciones con algunos valores atípicos para ilustrar cada método.

In [ ]:
np.random.seed(42)

# Datos normales: montos entre $50 y $500, distribución aproximadamente normal
montos_normales = np.random.normal(loc=250, scale=60, size=200)

# Anomalías: montos muy altos o muy bajos
anomalias_altas = np.array([950, 1100, 870, 980])
anomalias_bajas = np.array([-50, -80])   # Reembolsos inválidos

# Unir todo
montos = np.concatenate([montos_normales, anomalias_altas, anomalias_bajas])
np.random.shuffle(montos)   # Mezclar para que no estén al final

print(f'Total de observaciones : {len(montos)}')
print(f'Media                  : {montos.mean():.2f}')
print(f'Desviación estándar    : {montos.std():.2f}')
print(f'Mínimo / Máximo        : {montos.min():.2f} / {montos.max():.2f}')

### 3.1 Método Z-score

El **Z-score** mide cuántas desviaciones estándar se aleja un punto de la media:

$$z_i = \frac{x_i - \bar{x}}{s}$$

Un umbral típico es $|z| > 3$, correspondiente aproximadamente al 0.27% de los datos en una distribución normal.

**Supuesto:** los datos siguen una distribución aproximadamente normal.  
**Limitación:** la media y la desviación estándar son sensibles a los propios outliers, lo que puede enmascarar anomalías extremas (*swamping* y *masking*).

In [ ]:
from scipy import stats

# Calcular Z-scores
z_scores = np.abs(stats.zscore(montos))

# Umbral estándar
umbral_z = 3.0
outliers_z = montos[z_scores > umbral_z]

print(f'Umbral Z-score         : |z| > {umbral_z}')
print(f'Anomalías detectadas   : {len(outliers_z)}')
print(f'Valores anómalos       : {np.sort(outliers_z)}')

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histograma con umbral
ax = axes[0]
ax.hist(montos, bins=30, color='steelblue', alpha=0.7, edgecolor='white')
limite_sup = montos.mean() + umbral_z * montos.std()
limite_inf = montos.mean() - umbral_z * montos.std()
ax.axvline(limite_sup, color='red', linestyle='--', linewidth=2,
           label=f'Umbral superior (+{umbral_z}σ) = {limite_sup:.0f}')
ax.axvline(limite_inf, color='orange', linestyle='--', linewidth=2,
           label=f'Umbral inferior (-{umbral_z}σ) = {limite_inf:.0f}')
ax.set_title('Distribución de Montos — Z-score')
ax.set_xlabel('Monto ($)')
ax.set_ylabel('Frecuencia')
ax.legend(fontsize=9)

# Z-scores a lo largo del índice
ax = axes[1]
colores_z = ['red' if z > umbral_z else 'steelblue' for z in z_scores]
ax.scatter(range(len(montos)), z_scores, c=colores_z, alpha=0.6, s=20)
ax.axhline(umbral_z, color='red', linestyle='--', linewidth=2,
           label=f'Umbral = {umbral_z}')
ax.set_title('Z-scores por Observación')
ax.set_xlabel('Índice')
ax.set_ylabel('|Z-score|')
ax.legend()

plt.tight_layout()
plt.show()

### 3.2 Método IQR (Rango Intercuartílico)

El método IQR es **robusto** a los propios outliers porque utiliza percentiles en lugar de la media:

$$\text{IQR} = Q_3 - Q_1$$

Se declara anómalo cualquier punto que caiga fuera de las **vallas de Tukey**:

$$\text{Límite inferior} = Q_1 - k \cdot \text{IQR}$$
$$\text{Límite superior} = Q_3 + k \cdot \text{IQR}$$

donde $k = 1.5$ es el valor estándar (para vallas internas) y $k = 3.0$ para vallas externas (anomalías extremas).

Esta es la misma lógica que usa el **diagrama de caja** (*boxplot*).

In [ ]:
# Calcular cuartiles e IQR
Q1 = np.percentile(montos, 25)
Q3 = np.percentile(montos, 75)
IQR = Q3 - Q1

k = 1.5  # Factor estándar de Tukey
lim_inf_iqr = Q1 - k * IQR
lim_sup_iqr = Q3 + k * IQR

mascara_outlier_iqr = (montos < lim_inf_iqr) | (montos > lim_sup_iqr)
outliers_iqr = montos[mascara_outlier_iqr]

print(f'Q1 = {Q1:.2f}  |  Q3 = {Q3:.2f}  |  IQR = {IQR:.2f}')
print(f'Límite inferior (Q1 - 1.5·IQR) : {lim_inf_iqr:.2f}')
print(f'Límite superior (Q3 + 1.5·IQR) : {lim_sup_iqr:.2f}')
print(f'Anomalías detectadas            : {len(outliers_iqr)}')
print(f'Valores anómalos                : {np.sort(outliers_iqr)}')

# Visualización: boxplot + strip
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
bp = ax.boxplot(montos, vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightsteelblue'),
                medianprops=dict(color='navy', linewidth=2),
                flierprops=dict(marker='o', markerfacecolor='red',
                                markersize=8, linestyle='none'))
ax.set_title('Boxplot — Método IQR')
ax.set_ylabel('Monto ($)')
ax.set_xticks([])

# Distribución coloreada
ax = axes[1]
ax.hist(montos[~mascara_outlier_iqr], bins=25, color='steelblue',
        alpha=0.7, label='Normal', edgecolor='white')
ax.hist(montos[mascara_outlier_iqr], bins=5, color='red',
        alpha=0.8, label='Outlier IQR', edgecolor='white')
ax.axvline(lim_inf_iqr, color='orange', linestyle='--', linewidth=2,
           label=f'Q1 - 1.5·IQR = {lim_inf_iqr:.0f}')
ax.axvline(lim_sup_iqr, color='red', linestyle='--', linewidth=2,
           label=f'Q3 + 1.5·IQR = {lim_sup_iqr:.0f}')
ax.set_title('Histograma con Vallas IQR')
ax.set_xlabel('Monto ($)')
ax.set_ylabel('Frecuencia')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

### 3.3 Prueba de Grubbs

La **prueba de Grubbs** (también llamada prueba del valor extremo estudentizado) es un test estadístico formal para detectar **un único outlier** en un conjunto de datos que sigue una distribución normal.

El estadístico de Grubbs es:

$$G = \frac{\max_i |x_i - \bar{x}|}{s}$$

Se rechaza la hipótesis nula (no hay outlier) si $G$ supera el valor crítico basado en la distribución $t$ de Student:

$$G_{\text{crítico}} = \frac{N-1}{\sqrt{N}} \sqrt{\frac{t^2_{\alpha/(2N),N-2}}{N - 2 + t^2_{\alpha/(2N),N-2}}}$$

**Limitación:** detecta un outlier a la vez; requiere aplicación iterativa (prueba de Grubbs secuencial) si se sospecha de varios.

In [ ]:
from scipy.stats import t as t_dist

def grubbs_test(datos, alpha=0.05):
    """
    Prueba de Grubbs para detectar un outlier en 'datos'.
    Retorna: (estadístico G, valor crítico, índice del candidato, es_outlier)
    """
    n = len(datos)
    media = np.mean(datos)
    s = np.std(datos, ddof=1)  # desviación estándar muestral

    # Estadístico G: valor más alejado de la media
    G = np.max(np.abs(datos - media)) / s
    idx_candidato = np.argmax(np.abs(datos - media))

    # Valor crítico
    t_critico = t_dist.ppf(1 - alpha / (2 * n), df=n - 2)
    G_critico = ((n - 1) / np.sqrt(n)) * np.sqrt(
        t_critico**2 / (n - 2 + t_critico**2)
    )

    es_outlier = G > G_critico
    return G, G_critico, idx_candidato, es_outlier


def grubbs_iterativo(datos, alpha=0.05, max_iter=10):
    """
    Aplica la prueba de Grubbs de forma iterativa para encontrar
    múltiples outliers, eliminando uno por iteración.
    """
    datos_trabajo = datos.copy()
    indices_originales = list(range(len(datos)))
    outliers_encontrados = []

    for it in range(max_iter):
        G, G_crit, idx, es_outlier = grubbs_test(datos_trabajo, alpha)
        if not es_outlier:
            break
        idx_original = indices_originales[idx]
        outliers_encontrados.append((idx_original, datos_trabajo[idx], G, G_crit))
        # Eliminar el outlier detectado
        datos_trabajo = np.delete(datos_trabajo, idx)
        indices_originales.pop(idx)

    return outliers_encontrados


# Aplicar prueba iterativa
resultados_grubbs = grubbs_iterativo(montos, alpha=0.05)

print('=== Prueba de Grubbs (iterativa, α = 0.05) ===')
print(f'{'Iteración':<12} {'Índice':<8} {'Valor':<12} {'G':<10} {'G_crítico':<12} Decisión')
print('-' * 65)
for i, (idx, val, G, G_crit) in enumerate(resultados_grubbs, 1):
    decision = 'OUTLIER' if G > G_crit else 'Normal'
    print(f'{i:<12} {idx:<8} {val:<12.2f} {G:<10.4f} {G_crit:<12.4f} {decision}')

if not resultados_grubbs:
    print('No se detectaron outliers.')

### 3.4 Comparación de métodos unidimensionales

Resumimos los outliers detectados por cada método.

In [ ]:
# Reunir máscaras de cada método
mascara_z = z_scores > umbral_z
mascara_iqr = mascara_outlier_iqr
indices_grubbs = {idx for idx, *_ in resultados_grubbs}
mascara_grubbs = np.array([i in indices_grubbs for i in range(len(montos))])

# DataFrame resumen
df_resumen_1d = pd.DataFrame({
    'Monto': montos,
    'Z-score': z_scores.round(3),
    'Outlier_Zscore': mascara_z,
    'Outlier_IQR': mascara_iqr,
    'Outlier_Grubbs': mascara_grubbs
})

# Mostrar solo los puntos marcados por al menos un método
outliers_union = df_resumen_1d[
    df_resumen_1d['Outlier_Zscore'] |
    df_resumen_1d['Outlier_IQR'] |
    df_resumen_1d['Outlier_Grubbs']
].sort_values('Monto', ascending=False)

print(f'Puntos marcados como outlier por al menos un método: {len(outliers_union)}')
print()
print(outliers_union.to_string(index=True))

print('\nResumen por método:')
print(f'  Z-score (|z|>3) : {mascara_z.sum()} outliers')
print(f'  IQR (k=1.5)     : {mascara_iqr.sum()} outliers')
print(f'  Grubbs (α=0.05) : {mascara_grubbs.sum()} outliers')

---
## 4. Algoritmos Multidimensionales (Sesión 20)

Cuando trabajamos con múltiples variables simultáneamente, los métodos unidimensionales pierden poder: un punto puede ser perfectamente normal en cada dimensión individual pero anómalo en su **combinación**. Necesitamos detectores que capturen la estructura conjunta de los datos.

Generaremos un conjunto de datos sintético bidimensional para visualizar cómo funciona cada algoritmo.

In [ ]:
from sklearn.datasets import make_blobs

np.random.seed(42)

# Datos normales: dos clústeres de transacciones regulares
X_normal, _ = make_blobs(n_samples=300, centers=[[2, 2], [6, 7]],
                          cluster_std=0.8, random_state=42)

# Anomalías: puntos dispersos en todo el espacio
X_anomaly = np.random.uniform(low=[-2, -2], high=[10, 12], size=(18, 2))

X_2d = np.vstack([X_normal, X_anomaly])
y_2d = np.array([0] * len(X_normal) + [1] * len(X_anomaly))  # 0=normal, 1=anomalía

# Porcentaje de anomalías (contamination)
contam = len(X_anomaly) / len(X_2d)
print(f'Total de puntos   : {len(X_2d)}')
print(f'Normales          : {len(X_normal)}')
print(f'Anomalías         : {len(X_anomaly)}')
print(f'Contaminación     : {contam:.3f} ({contam*100:.1f}%)')

# Visualización inicial
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(X_normal[:, 0], X_normal[:, 1], c='steelblue',
           alpha=0.6, s=30, label='Normal')
ax.scatter(X_anomaly[:, 0], X_anomaly[:, 1], c='red',
           marker='*', s=150, zorder=5, label='Anomalía')
ax.set_title('Dataset sintético 2D — Vista general')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend()
plt.tight_layout()
plt.show()

### 4.1 Isolation Forest

**Intuición:** Las anomalías son pocas y diferentes. Por lo tanto, son más fáciles de *aislar* que los puntos normales mediante particiones aleatorias del espacio de características.

**Funcionamiento:**
1. Se construyen $T$ árboles de aislamiento (*isolation trees*). Cada árbol particiona el espacio de características seleccionando aleatoriamente una variable y un punto de corte.
2. Para cada punto $x$, se mide la **longitud de ruta promedio** $h(x)$: cuántas particiones se necesitan para aislarlo.
3. Las anomalías se aíslan en **pocos pasos** (rutas cortas) porque están en regiones de baja densidad.

**Puntuación de anomalía:**
$$s(x, n) = 2^{-\frac{E[h(x)]}{c(n)}}$$

donde $c(n)$ es la longitud de ruta promedio de un árbol de búsqueda binaria fallido para $n$ muestras. Un score cercano a 1 indica anomalía; cercano a 0.5 indica punto normal.

**Ventajas:** escala bien con datos de alta dimensión, no requiere distancias, maneja múltiples clústeres.

In [ ]:
from sklearn.ensemble import IsolationForest

# Entrenar Isolation Forest
iso_forest = IsolationForest(
    n_estimators=200,      # número de árboles
    contamination=contam,  # fracción esperada de anomalías
    random_state=42
)
iso_forest.fit(X_2d)

# Predicciones: sklearn usa 1=normal, -1=anomalía
pred_if = iso_forest.predict(X_2d)                 # etiquetas
scores_if = iso_forest.decision_function(X_2d)     # score de anomalía (mayor = más normal)

# Convertir a convención 0/1 (0=normal, 1=anomalía)
pred_if_01 = (pred_if == -1).astype(int)

print(f'Anomalías detectadas por Isolation Forest: {pred_if_01.sum()}')
print(f'Anomalías reales                         : {y_2d.sum()}')

# Visualizar frontera de decisión
xx, yy = np.meshgrid(
    np.linspace(X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1, 200),
    np.linspace(X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1, 200)
)
Z_if = iso_forest.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z_if = Z_if.reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contourf(xx, yy, Z_if, levels=20, cmap='RdYlGn', alpha=0.5)
plt.colorbar(cs, ax=ax, label='Anomaly score (mayor = más normal)')
ax.contour(xx, yy, Z_if, levels=[0], colors='black',
           linewidths=2, linestyles='--')

# Puntos coloreados por predicción
correctos = pred_if_01 == y_2d
ax.scatter(X_2d[correctos & (y_2d == 0), 0],
           X_2d[correctos & (y_2d == 0), 1],
           c='steelblue', alpha=0.7, s=30, label='Normal (correcto)')
ax.scatter(X_2d[correctos & (y_2d == 1), 0],
           X_2d[correctos & (y_2d == 1), 1],
           c='red', marker='*', s=180, label='Anomalía detectada')
ax.scatter(X_2d[~correctos, 0], X_2d[~correctos, 1],
           c='orange', marker='x', s=100, linewidth=2, label='Error de clasificación')

ax.set_title('Isolation Forest — Frontera de decisión')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 4.2 Local Outlier Factor (LOF)

**Intuición:** Un punto es anómalo si su **densidad local** es significativamente menor que la de sus vecinos. A diferencia de Isolation Forest, LOF es un método basado en densidad que detecta outliers locales (no solo globales).

**Funcionamiento:**
1. Para cada punto $p$, se identifican sus $k$ vecinos más cercanos.
2. Se calcula la **distancia de alcanzabilidad** (*reachability distance*):  
   $\text{reach-dist}_k(p, o) = \max(d_k(o),\, d(p, o))$
3. Se calcula la **densidad de alcanzabilidad local** (*local reachability density*):  
   $\text{lrd}_k(p) = \frac{1}{\sum_{o \in N_k(p)} \text{reach-dist}_k(p,o) / |N_k(p)|}$
4. El **LOF score** es la razón entre la densidad de los vecinos y la densidad de $p$:  
   $\text{LOF}_k(p) = \frac{\sum_{o \in N_k(p)} \text{lrd}_k(o) / \text{lrd}_k(p)}{|N_k(p)|}$

Un LOF $\approx 1$ indica punto normal; LOF $\gg 1$ indica anomalía.

**Ventajas:** detecta anomalías en regiones de densidad variable (importante cuando hay múltiples clústeres de diferente densidad).

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

# Entrenar LOF
lof = LocalOutlierFactor(
    n_neighbors=20,          # k vecinos
    contamination=contam,    # fracción esperada de anomalías
    algorithm='auto',
    metric='euclidean'
)

# LOF en sklearn solo soporta fit_predict (no predict separado en modo novelty=False)
pred_lof = lof.fit_predict(X_2d)  # 1=normal, -1=anomalía
scores_lof = -lof.negative_outlier_factor_  # Mayor score = más anómalo

# Convertir a 0/1
pred_lof_01 = (pred_lof == -1).astype(int)

print(f'Anomalías detectadas por LOF   : {pred_lof_01.sum()}')
print(f'Anomalías reales               : {y_2d.sum()}')

# Visualización con tamaño de punto proporcional al LOF score
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel izquierdo: predicciones
ax = axes[0]
correctos_lof = pred_lof_01 == y_2d
ax.scatter(X_2d[correctos_lof & (y_2d == 0), 0],
           X_2d[correctos_lof & (y_2d == 0), 1],
           c='steelblue', alpha=0.7, s=30, label='Normal (correcto)')
ax.scatter(X_2d[correctos_lof & (y_2d == 1), 0],
           X_2d[correctos_lof & (y_2d == 1), 1],
           c='red', marker='*', s=180, label='Anomalía detectada')
ax.scatter(X_2d[~correctos_lof, 0], X_2d[~correctos_lof, 1],
           c='orange', marker='x', s=100, linewidth=2, label='Error')
ax.set_title('LOF — Predicciones')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')
ax.legend(fontsize=9)

# Panel derecho: LOF scores (tamaño proporcional al outlier factor)
ax = axes[1]
sc = ax.scatter(X_2d[:, 0], X_2d[:, 1],
                c=scores_lof, cmap='YlOrRd',
                s=np.clip(scores_lof * 20, 10, 200),
                alpha=0.8, edgecolors='gray', linewidths=0.3)
plt.colorbar(sc, ax=ax, label='LOF score (mayor = más anómalo)')
ax.set_title('LOF — Mapa de scores (tamaño ∝ LOF score)')
ax.set_xlabel('Feature 1')
ax.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

### 4.3 One-Class SVM (OCSVM)

**Intuición:** Aprender una frontera de decisión que encierre la región donde se concentra la mayoría de los datos de entrenamiento (asumiendo que todos son normales). Cualquier punto fuera de esa frontera se declara anomalía.

**Funcionamiento:**
- Es una extensión de SVM que usa la idea del hiperplano de separación, pero en lugar de separar dos clases, **separa los datos del origen** en el espacio de características inducido por un kernel.
- Con kernel RBF, la frontera puede ser no lineal y adaptarse a formas complejas.
- El parámetro $\nu \in (0,1]$ controla el límite superior de la fracción de outliers de entrenamiento y el límite inferior de vectores soporte.

**Ventajas:** potente con kernels no lineales; útil cuando solo se dispone de datos normales para entrenar.

**Desventajas:** costoso computacionalmente en datos grandes; sensible a la elección del kernel y parámetros.

In [ ]:
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler

# OCSVM es sensible a la escala — es importante normalizar
scaler_2d = StandardScaler()
X_2d_scaled = scaler_2d.fit_transform(X_2d)

# Entrenar OCSVM
ocsvm = OneClassSVM(
    kernel='rbf',
    gamma='scale',   # gamma = 1 / (n_features * Var(X))
    nu=contam        # aproximadamente = fracción de outliers
)
ocsvm.fit(X_2d_scaled)

# Predicciones: 1=normal, -1=anomalía
pred_ocsvm = ocsvm.predict(X_2d_scaled)
scores_ocsvm = ocsvm.decision_function(X_2d_scaled)  # Mayor = más normal

# Convertir a 0/1
pred_ocsvm_01 = (pred_ocsvm == -1).astype(int)

print(f'Anomalías detectadas por OCSVM : {pred_ocsvm_01.sum()}')
print(f'Anomalías reales               : {y_2d.sum()}')

# Visualización de la frontera de decisión
xx_s, yy_s = np.meshgrid(
    np.linspace(X_2d_scaled[:, 0].min() - 0.5, X_2d_scaled[:, 0].max() + 0.5, 300),
    np.linspace(X_2d_scaled[:, 1].min() - 0.5, X_2d_scaled[:, 1].max() + 0.5, 300)
)
Z_ocsvm = ocsvm.decision_function(np.c_[xx_s.ravel(), yy_s.ravel()])
Z_ocsvm = Z_ocsvm.reshape(xx_s.shape)

fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contourf(xx_s, yy_s, Z_ocsvm, levels=20, cmap='RdYlGn', alpha=0.5)
plt.colorbar(cs, ax=ax, label='Decision function (mayor = más normal)')
ax.contour(xx_s, yy_s, Z_ocsvm, levels=[0], colors='black',
           linewidths=2, linestyles='--')

correctos_oc = pred_ocsvm_01 == y_2d
ax.scatter(X_2d_scaled[correctos_oc & (y_2d == 0), 0],
           X_2d_scaled[correctos_oc & (y_2d == 0), 1],
           c='steelblue', alpha=0.7, s=30, label='Normal (correcto)')
ax.scatter(X_2d_scaled[correctos_oc & (y_2d == 1), 0],
           X_2d_scaled[correctos_oc & (y_2d == 1), 1],
           c='red', marker='*', s=180, label='Anomalía detectada')
ax.scatter(X_2d_scaled[~correctos_oc, 0], X_2d_scaled[~correctos_oc, 1],
           c='orange', marker='x', s=100, linewidth=2, label='Error')

ax.set_title('One-Class SVM (kernel RBF) — Frontera de decisión')
ax.set_xlabel('Feature 1 (escalada)')
ax.set_ylabel('Feature 2 (escalada)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### 4.4 Uso de `pyod` (librería especializada)

`pyod` (*Python Outlier Detection*) es la librería de referencia para detección de anomalías en Python. Ofrece más de 40 algoritmos con una API unificada compatible con scikit-learn.

La siguiente celda intenta importar `pyod` y, si no está instalada, cae elegantemente al equivalente de `sklearn`.

In [ ]:
try:
    from pyod.models.iforest import IForest as PyodIForest
    from pyod.models.lof import LOF as PyodLOF
    from pyod.models.ocsvm import OCSVM as PyodOCSVM
    PYOD_DISPONIBLE = True
    print('pyod disponible — usando PyOD para la sección de comparación.')
except ImportError:
    PYOD_DISPONIBLE = False
    print('pyod no instalado — usando equivalentes de sklearn.')
    print('Para instalar: pip install pyod')

# Nota: en ambos casos el análisis continúa correctamente.

---
## 5. Evaluación de Detectores de Anomalías

### ¿Por qué no usar accuracy?

En un dataset con 1% de anomalías, un clasificador que siempre predice «normal» tendría 99% de accuracy pero **0% de utilidad práctica**. Necesitamos métricas que reflejen la capacidad del detector de encontrar las pocas anomalías reales.

### Métricas clave

Definiendo: $TP$ = anomalías detectadas correctamente, $FP$ = falsos positivos, $FN$ = anomalías no detectadas:

| Métrica | Fórmula | Qué mide |
|---|---|---|
| **Precisión** | $TP / (TP + FP)$ | De las alertas generadas, ¿cuántas son reales? |
| **Recall** | $TP / (TP + FN)$ | De las anomalías reales, ¿cuántas se detectaron? |
| **F1-score** | $2 \cdot P \cdot R / (P + R)$ | Media armónica entre precisión y recall |
| **ROC-AUC** | Área bajo la curva ROC | Capacidad general de ranking del detector |

**Precision-Recall AUC** suele ser más informativo que ROC-AUC cuando las clases están muy desbalanceadas.

In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    RocCurveDisplay, PrecisionRecallDisplay
)

def evaluar_detector(nombre, y_verdadero, y_pred, y_scores):
    """
    Calcula y muestra las métricas de evaluación para un detector de anomalías.
    
    Parámetros
    ----------
    nombre      : str — nombre del detector
    y_verdadero : array — etiquetas reales (0=normal, 1=anomalía)
    y_pred      : array — predicciones binarias del detector
    y_scores    : array — puntuaciones de anomalía (mayor = más anómalo)
    """
    precision = precision_score(y_verdadero, y_pred, zero_division=0)
    recall    = recall_score(y_verdadero, y_pred, zero_division=0)
    f1        = f1_score(y_verdadero, y_pred, zero_division=0)
    roc_auc   = roc_auc_score(y_verdadero, y_scores)
    pr_auc    = average_precision_score(y_verdadero, y_scores)

    return {
        'Detector': nombre,
        'Precisión': round(precision, 4),
        'Recall': round(recall, 4),
        'F1-score': round(f1, 4),
        'ROC-AUC': round(roc_auc, 4),
        'PR-AUC': round(pr_auc, 4)
    }


# Scores normalizados para comparación (mayor = más anómalo)
# Isolation Forest: invertimos el signo (sklearn: mayor score = más normal)
scores_if_norm = -scores_if
scores_ocsvm_norm = -scores_ocsvm
# LOF ya está en convención mayor = más anómalo

# Calcular métricas para los tres detectores
resultados_2d = [
    evaluar_detector('Isolation Forest', y_2d, pred_if_01, scores_if_norm),
    evaluar_detector('LOF (k=20)', y_2d, pred_lof_01, scores_lof),
    evaluar_detector('One-Class SVM', y_2d, pred_ocsvm_01, scores_ocsvm_norm),
]

df_resultados_2d = pd.DataFrame(resultados_2d).set_index('Detector')
print('=== Comparación de detectores — Dataset sintético 2D ===')
print(df_resultados_2d.to_string())

In [ ]:
# Curvas ROC y Precision-Recall para los tres detectores
from sklearn.metrics import roc_curve, precision_recall_curve

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

detectores_plot = [
    ('Isolation Forest', scores_if_norm, pred_if_01),
    ('LOF (k=20)',       scores_lof,    pred_lof_01),
    ('One-Class SVM',    scores_ocsvm_norm, pred_ocsvm_01),
]
colores_plot = ['steelblue', 'darkorange', 'forestgreen']

# --- Curva ROC ---
ax = axes[0]
for (nombre, scores, _), color in zip(detectores_plot, colores_plot):
    fpr, tpr, _ = roc_curve(y_2d, scores)
    auc_val = roc_auc_score(y_2d, scores)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f'{nombre} (AUC = {auc_val:.3f})')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aleatorio')
ax.set_xlabel('Tasa de Falsos Positivos')
ax.set_ylabel('Tasa de Verdaderos Positivos (Recall)')
ax.set_title('Curva ROC')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

# --- Curva Precision-Recall ---
ax = axes[1]
for (nombre, scores, _), color in zip(detectores_plot, colores_plot):
    prec, rec, _ = precision_recall_curve(y_2d, scores)
    pr_auc_val = average_precision_score(y_2d, scores)
    ax.plot(rec, prec, color=color, linewidth=2,
            label=f'{nombre} (AP = {pr_auc_val:.3f})')
baseline = y_2d.mean()  # tasa de anomalías
ax.axhline(baseline, color='black', linestyle='--', linewidth=1,
           label=f'Línea base ({baseline:.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precisión')
ax.set_title('Curva Precision-Recall')
ax.legend(fontsize=9)
ax.grid(alpha=0.3)

plt.suptitle('Evaluación de Detectores', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 5.1 El parámetro de contaminación (*contamination*)

El parámetro `contamination` le indica al detector la **fracción esperada de anomalías** en los datos. Es crítico porque:

- Define el umbral de decisión: el detector marca como anómalos los `contamination * n` puntos con mayor score.
- Si se sobreestima, aumentan los falsos positivos (alertas innecesarias).
- Si se subestima, aumentan los falsos negativos (anomalías no detectadas).

En la práctica, este valor debe estimarse a partir del conocimiento del dominio o mediante validación cruzada.

In [ ]:
# Efecto del parámetro contamination en Isolation Forest
contaminaciones = [0.02, 0.05, 0.10, 0.20, 0.30]
verdadera_contam = contam  # la real

resultados_contam = []
for c in contaminaciones:
    clf = IsolationForest(n_estimators=100, contamination=c, random_state=42)
    clf.fit(X_2d)
    pred = (clf.predict(X_2d) == -1).astype(int)
    sc = -clf.decision_function(X_2d)
    resultados_contam.append({
        'Contamination': c,
        'Anomalías predichas': pred.sum(),
        'Precisión': round(precision_score(y_2d, pred, zero_division=0), 3),
        'Recall': round(recall_score(y_2d, pred, zero_division=0), 3),
        'F1-score': round(f1_score(y_2d, pred, zero_division=0), 3),
    })

df_contam = pd.DataFrame(resultados_contam)
print(f'Contaminación real : {verdadera_contam:.3f}')
print()
print(df_contam.to_string(index=False))

---
## 6. Caso Práctico Completo: Detección de Fraude Financiero

Construiremos un pipeline completo sobre un dataset sintético que simula transacciones financieras, con características similares a los datasets de fraude reales:

- **Variables de comportamiento:** monto, hora, frecuencia de transacciones
- **Variables de perfil:** antigüedad del cliente, saldo promedio
- **Clase:** 0 = transacción legítima, 1 = fraude (≈ 2% del total)

### 6.1 Generación del dataset sintético

In [ ]:
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

np.random.seed(42)

# -------------------------------------------------------
# Generamos un dataset de clasificación binaria desbalanceada
# que simula transacciones financieras
# -------------------------------------------------------
N_TOTAL = 5000
TASA_FRAUDE = 0.02   # 2% de fraude — típico en datos reales
N_FRAUDE = int(N_TOTAL * TASA_FRAUDE)
N_LEGIT  = N_TOTAL - N_FRAUDE

# make_classification genera features con estructura realista
X_raw, y_raw = make_classification(
    n_samples=N_TOTAL,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    n_repeated=0,
    n_classes=2,
    weights=[1 - TASA_FRAUDE, TASA_FRAUDE],  # clase desbalanceada
    flip_y=0.01,
    random_state=42
)

# Renombrar columnas para el contexto financiero
nombres_features = [
    'monto_log',        # log del monto de la transacción
    'hora_normalizada', # hora del día normalizada
    'frec_7dias',       # frecuencia de transacciones últimos 7 días
    'dist_comercio',    # distancia al comercio habitual
    'saldo_promedio',   # saldo promedio mensual
    'antiguedad_años',  # años como cliente
    'ratio_monto',      # ratio monto / promedio histórico
    'n_paises_30d',     # países distintos últimos 30 días
    'frec_declinadas',  # transacciones declinadas recientes
    'score_riesgo'      # score de riesgo interno del banco
]

df_fraude = pd.DataFrame(X_raw, columns=nombres_features)
df_fraude['fraude'] = y_raw

print(f'Dataset generado:')
print(f'  Total de transacciones : {N_TOTAL:,}')
print(f'  Legítimas              : {(y_raw == 0).sum():,} ({(y_raw==0).mean()*100:.1f}%)')
print(f'  Fraudulentas           : {(y_raw == 1).sum():,} ({(y_raw==1).mean()*100:.1f}%)')
print()
print('Primeras 5 filas:')
df_fraude.head()

### 6.2 Análisis exploratorio rápido

In [ ]:
# Estadísticas por clase
print('=== Estadísticas por clase ===')
print(df_fraude.groupby('fraude')[nombres_features[:6]].mean().round(3).to_string())

# Distribución de las features más informativas
features_a_plotear = ['monto_log', 'frec_7dias', 'dist_comercio', 'ratio_monto']

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
axes = axes.ravel()

for i, feat in enumerate(features_a_plotear):
    ax = axes[i]
    ax.hist(df_fraude.loc[df_fraude['fraude']==0, feat],
            bins=40, alpha=0.6, color='steelblue',
            density=True, label='Legítima')
    ax.hist(df_fraude.loc[df_fraude['fraude']==1, feat],
            bins=40, alpha=0.6, color='red',
            density=True, label='Fraude')
    ax.set_title(f'Distribución: {feat}')
    ax.set_xlabel(feat)
    ax.set_ylabel('Densidad')
    ax.legend(fontsize=9)

plt.suptitle('Distribución de Features por Clase', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 6.3 Preprocesamiento y División train/test

In [ ]:
# Separar features y etiquetas
X_fraude = df_fraude.drop('fraude', axis=1).values   # CORRECTO: usar axis=1
y_fraude = df_fraude['fraude'].values

# División estratificada train (80%) / test (20%)
# Estratificada para mantener la proporción de fraude en ambos conjuntos
X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fraude, y_fraude,
    test_size=0.20,
    random_state=42,
    stratify=y_fraude   # mantiene la proporción 98/2 en train y test
)

# Escalar features (importante para LOF y OCSVM)
scaler_f = StandardScaler()
X_train_f_sc = scaler_f.fit_transform(X_train_f)   # fit solo en train
X_test_f_sc  = scaler_f.transform(X_test_f)         # transform en test

print(f'Train : {X_train_f.shape[0]:,} muestras | Fraude: {y_train_f.sum():,} ({y_train_f.mean()*100:.1f}%)')
print(f'Test  : {X_test_f.shape[0]:,} muestras  | Fraude: {y_test_f.sum():,} ({y_test_f.mean()*100:.1f}%)')

### 6.4 Entrenamiento y evaluación de los tres detectores

Un punto importante: los detectores de anomalías son métodos **no supervisados**. Los entrenamos **solo con X** (sin usar las etiquetas de fraude) y luego evaluamos qué tan bien separan los fraudes de las transacciones legítimas.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM

CONTAM_FRAUDE = TASA_FRAUDE  # usamos la tasa real como parámetro

# ---- Isolation Forest ----
print('Entrenando Isolation Forest...')
if_fraud = IsolationForest(
    n_estimators=300,
    contamination=CONTAM_FRAUDE,
    max_samples='auto',
    random_state=42,
    n_jobs=-1
)
if_fraud.fit(X_train_f_sc)  # solo X, sin etiquetas
pred_if_fraud = (if_fraud.predict(X_test_f_sc) == -1).astype(int)
scores_if_fraud = -if_fraud.decision_function(X_test_f_sc)

# ---- LOF ----
# Nota: LOF en sklearn no tiene método predict separado en modo transductivo
# Usamos novelty=True para poder predecir en datos nuevos
print('Entrenando LOF (novelty=True)...')
lof_fraud = LocalOutlierFactor(
    n_neighbors=30,
    contamination=CONTAM_FRAUDE,
    novelty=True,   # permite predecir en datos nuevos
    n_jobs=-1
)
lof_fraud.fit(X_train_f_sc)
pred_lof_fraud = (lof_fraud.predict(X_test_f_sc) == -1).astype(int)
scores_lof_fraud = -lof_fraud.decision_function(X_test_f_sc)

# ---- One-Class SVM ----
print('Entrenando One-Class SVM (puede tardar unos segundos)...')
# Submuestreo para OCSVM (es O(n²) en memoria)
idx_subsample = np.random.choice(len(X_train_f_sc),
                                  size=min(2000, len(X_train_f_sc)),
                                  replace=False)
ocsvm_fraud = OneClassSVM(
    kernel='rbf',
    gamma='scale',
    nu=CONTAM_FRAUDE
)
ocsvm_fraud.fit(X_train_f_sc[idx_subsample])
pred_ocsvm_fraud = (ocsvm_fraud.predict(X_test_f_sc) == -1).astype(int)
scores_ocsvm_fraud = -ocsvm_fraud.decision_function(X_test_f_sc)

print('\nEntrenamiento completado.')

In [ ]:
# Evaluar los tres detectores en el conjunto de prueba
resultados_fraude = [
    evaluar_detector('Isolation Forest', y_test_f, pred_if_fraud, scores_if_fraud),
    evaluar_detector('LOF (k=30)', y_test_f, pred_lof_fraud, scores_lof_fraud),
    evaluar_detector('One-Class SVM', y_test_f, pred_ocsvm_fraud, scores_ocsvm_fraud),
]

df_resultados_fraude = pd.DataFrame(resultados_fraude).set_index('Detector')

print('=== Evaluación en Dataset de Fraude Sintético (conjunto de prueba) ===')
print(f'Tasa de fraude real: {y_test_f.mean()*100:.1f}%')
print()
print(df_resultados_fraude.to_string())

# Resaltar el mejor F1
mejor_f1_idx = df_resultados_fraude['F1-score'].idxmax()
print(f'\nMejor F1-score: {mejor_f1_idx} '
      f'(F1 = {df_resultados_fraude.loc[mejor_f1_idx, "F1-score"]})')

In [ ]:
# Visualización final: curvas ROC, PR y matrices de confusión
from sklearn.metrics import ConfusionMatrixDisplay

fig = plt.figure(figsize=(16, 10))

detectores_fraude = [
    ('Isolation Forest', scores_if_fraud, pred_if_fraud),
    ('LOF (k=30)',       scores_lof_fraud, pred_lof_fraud),
    ('One-Class SVM',    scores_ocsvm_fraud, pred_ocsvm_fraud),
]
colores_f = ['steelblue', 'darkorange', 'forestgreen']

# --- ROC ---
ax_roc = fig.add_subplot(2, 3, 1)
for (nombre, scores, _), color in zip(detectores_fraude, colores_f):
    fpr, tpr, _ = roc_curve(y_test_f, scores)
    auc_val = roc_auc_score(y_test_f, scores)
    ax_roc.plot(fpr, tpr, color=color, linewidth=2,
                label=f'{nombre} ({auc_val:.3f})')
ax_roc.plot([0,1],[0,1],'k--', linewidth=1)
ax_roc.set_title('Curva ROC')
ax_roc.set_xlabel('FPR')
ax_roc.set_ylabel('TPR')
ax_roc.legend(fontsize=8)
ax_roc.grid(alpha=0.3)

# --- Precision-Recall ---
ax_pr = fig.add_subplot(2, 3, 2)
for (nombre, scores, _), color in zip(detectores_fraude, colores_f):
    prec, rec, _ = precision_recall_curve(y_test_f, scores)
    ap = average_precision_score(y_test_f, scores)
    ax_pr.plot(rec, prec, color=color, linewidth=2,
               label=f'{nombre} (AP={ap:.3f})')
ax_pr.axhline(y_test_f.mean(), color='black', linestyle='--',
              linewidth=1, label=f'Base ({y_test_f.mean():.3f})')
ax_pr.set_title('Curva Precision-Recall')
ax_pr.set_xlabel('Recall')
ax_pr.set_ylabel('Precisión')
ax_pr.legend(fontsize=8)
ax_pr.grid(alpha=0.3)

# --- Barras comparativas de métricas ---
ax_bar = fig.add_subplot(2, 3, 3)
metricas = ['Precisión', 'Recall', 'F1-score', 'ROC-AUC']
x = np.arange(len(metricas))
width = 0.25
for i, (nombre, _, _) in enumerate(detectores_fraude):
    valores = [df_resultados_fraude.loc[nombre, m] for m in metricas]
    ax_bar.bar(x + i * width, valores, width, label=nombre,
               color=colores_f[i], alpha=0.8)
ax_bar.set_xticks(x + width)
ax_bar.set_xticklabels(metricas, fontsize=9)
ax_bar.set_ylim(0, 1.05)
ax_bar.set_title('Comparación de Métricas')
ax_bar.legend(fontsize=8)
ax_bar.grid(axis='y', alpha=0.3)

# --- Matrices de confusión ---
for i, ((nombre, _, pred), color) in enumerate(zip(detectores_fraude, colores_f)):
    ax_cm = fig.add_subplot(2, 3, 4 + i)
    cm = confusion_matrix(y_test_f, pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Legítima', 'Fraude'])
    disp.plot(ax=ax_cm, colorbar=False, cmap='Blues')
    ax_cm.set_title(f'{nombre}', fontsize=10)
    ax_cm.set_xlabel('Predicción', fontsize=9)
    ax_cm.set_ylabel('Real', fontsize=9)

plt.suptitle('Resultados del Pipeline de Detección de Fraude',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

### 6.5 Análisis de los scores de anomalía

Los scores continuos son más informativos que las predicciones binarias: permiten rankear las transacciones por riesgo y establecer umbrales según las necesidades del negocio (p. ej., revisar el top 1% de mayor riesgo).

In [ ]:
# Distribución de scores por clase real
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (nombre, scores, _), color in zip(axes, detectores_fraude, colores_f):
    # Normalizar scores al rango [0, 1] para facilitar comparación
    s_min, s_max = scores.min(), scores.max()
    scores_norm_plot = (scores - s_min) / (s_max - s_min + 1e-9)

    ax.hist(scores_norm_plot[y_test_f == 0], bins=40,
            alpha=0.6, density=True, color='steelblue', label='Legítima')
    ax.hist(scores_norm_plot[y_test_f == 1], bins=20,
            alpha=0.7, density=True, color='red', label='Fraude')
    ax.set_title(nombre)
    ax.set_xlabel('Score de anomalía (normalizado)')
    ax.set_ylabel('Densidad')
    ax.legend(fontsize=9)

plt.suptitle('Distribución de Scores por Clase',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ---
# Tabla: top 20 transacciones con mayor score de anomalía (Isolation Forest)
print('=== Top 20 transacciones más sospechosas (Isolation Forest) ===')
df_test_scores = pd.DataFrame({
    'score_IF': scores_if_fraud,
    'score_LOF': scores_lof_fraud,
    'pred_IF': pred_if_fraud,
    'real': y_test_f
})
top20 = df_test_scores.nlargest(20, 'score_IF')[['score_IF', 'score_LOF', 'pred_IF', 'real']]
top20['Es fraude real'] = top20['real'].map({0: 'No', 1: 'SI'})
top20['IF detectó'] = top20['pred_IF'].map({0: 'No', 1: 'SI'})
print(top20[['score_IF', 'score_LOF', 'IF detectó', 'Es fraude real']].round(4).to_string())

---
## 7. Resumen y Conclusiones

### 7.1 Cuándo usar cada método

| Algoritmo | Mejor para | Limitaciones |
|---|---|---|
| **Z-score / IQR** | EDA rápido, una sola variable, datos normales | No multidimensional, supuestos distribucionales |
| **Grubbs** | Test estadístico formal, muestra pequeña | Solo un outlier por iteración, requiere normalidad |
| **Isolation Forest** | Alta dimensionalidad, datos grandes, múltiples clústeres | Menos efectivo con anomalías locales sutiles |
| **LOF** | Anomalías locales, densidades variables | Sensible a k, costoso en datos grandes, no generaliza bien |
| **One-Class SVM** | Solo datos normales disponibles, fronteras complejas | Costoso O(n²), sensible a escala y parámetros |

### 7.2 Checklist para un proyecto de detección de anomalías

1. **Definir qué es una anomalía** en el contexto del negocio (¿puntual, contextual, colectiva?).
2. **Estimar la tasa de contaminación** con expertos del dominio.
3. **Explorar con métodos 1D** (Z-score, IQR) antes de escalar a métodos multidimensionales.
4. **Escalar las features** antes de LOF y OCSVM.
5. **Evaluar con Precision-Recall AUC**, no con accuracy.
6. **Comparar múltiples detectores** — no existe un ganador universal.
7. **Involucrar expertos del dominio** para validar los outliers detectados.


In [ ]:
# ============================================================
# EJERCICIO PROPUESTO
# ============================================================
# 1. Modifique TASA_FRAUDE a 0.05 y repita el pipeline.
#    ¿Cómo cambian las métricas? ¿Qué detector se beneficia más?
#
# 2. Implemente el Z-score multivariado usando la distancia de
#    Mahalanobis para el dataset de fraude y compárelo con
#    Isolation Forest.
#    Pista: scipy.spatial.distance.mahalanobis
#
# 3. Explore el efecto del parámetro n_neighbors en LOF
#    (pruebe k = 5, 10, 20, 50) y grafique el F1-score resultante.
#
# 4. Construya un ensemble: marque como fraude solo los puntos
#    que al menos DOS de los tres detectores clasifiquen como
#    anómalos. ¿Mejora la precisión? ¿A qué costo en recall?
# ============================================================

print('Notebook completado exitosamente.')
print('Semana 10 — Detección de Anomalías')
print('Universidad de los Andes — Minería de Datos')

## Referencias

**Texto guía**
- Notas de clase del curso (disponibles en el repositorio).

**Bibliografía complementaria**
- Aggarwal, C. C. (2017). *Outlier Analysis* (Cap. 2–3). Springer.
- Liu, F. T., Ting, K. M., & Zhou, Z. H. (2008). Isolation forest. *ICDM 2008*, 413–422.
- Breunig, M. M., Kriegel, H. P., Ng, R. T., & Sander, J. (2000). LOF: Identifying density-based local outliers. *SIGMOD Record*, 29(2), 93–104.
- Schölkopf, B., Platt, J. C., Shawe-Taylor, J., Smola, A. J., & Williamson, R. C. (2001). Estimating the support of a high-dimensional distribution. *Neural Computation*, 13(7), 1443–1471.